# 01 — Exploratory Data Analysis
Covers both datasets:
- **TrashCan 1.0** (optical): class distribution, image size stats, sample visualization
- **FLS Sonar** (watertank-segmentation): class distribution, sonar image characteristics

Run this notebook **after** downloading both datasets.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from pycocotools.coco import COCO

DATA_ROOT      = Path('../marine-debris-fls-datasets/md_fls_dataset/data')
TRASHCAN_ROOT  = Path('../data/trashcan')   # not yet downloaded
SEG_ROOT       = DATA_ROOT / 'watertank-segmentation'
TURNTABLE_ROOT = DATA_ROOT / 'turntable-cropped'
WATERTANK_ROOT = DATA_ROOT / 'watertank-cropped'

FLS_CLASSES = [
    'background', 'bottle', 'can', 'chain', 'drink-carton',
    'hook', 'propeller', 'shampoo-bottle', 'standing-bottle', 'tire', 'valve', 'wall'
]

print('Seg root exists:      ', SEG_ROOT.exists())
print('Turntable root exists:', TURNTABLE_ROOT.exists())
print('Watertank root exists:', WATERTANK_ROOT.exists())

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)


Seg root exists:       True
Turntable root exists: True
Watertank root exists: True


## 1. TrashCan — Class Distribution

In [2]:
# Load COCO annotation files for train + val
splits = ['train', 'val', 'test']
coco_by_split = {}
for split in splits:
    ann_path = TRASHCAN_ROOT / 'annotations' / f'instances_{split}.json'
    if ann_path.exists():
        coco_by_split[split] = COCO(str(ann_path))
        print(f'{split}: {len(coco_by_split[split].imgs)} images, '
              f'{len(coco_by_split[split].anns)} annotations')
    else:
        print(f'[WARN] {ann_path} not found — skip')

[WARN] ../data/trashcan/annotations/instances_train.json not found — skip
[WARN] ../data/trashcan/annotations/instances_val.json not found — skip
[WARN] ../data/trashcan/annotations/instances_test.json not found — skip


In [3]:
# Count annotations per category per split
from collections import Counter

all_counts = {}
for split, coco in coco_by_split.items():
    cats = {c['id']: c['name'] for c in coco.loadCats(coco.getCatIds())}
    counts = Counter(cats[a['category_id']] for a in coco.anns.values())
    all_counts[split] = counts
    print(f'\n{split}: {dict(counts)}')

if not all_counts:
    print('[WARN] No TrashCan annotations found — skipping class distribution plot.')
else:
    fig, axes = plt.subplots(1, len(all_counts), figsize=(5 * len(all_counts), 4))
    if len(all_counts) == 1:
        axes = [axes]
    for ax, (split, counts) in zip(axes, all_counts.items()):
        bars = ax.bar(counts.keys(), counts.values(), color=['#2196F3', '#4CAF50', '#FF9800'])
        ax.bar_label(bars)
        ax.set_title(f'TrashCan — {split}')
        ax.set_ylabel('# annotations')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'trashcan_class_dist.png', dpi=150)
    plt.show()

    if 'train' in all_counts:
        c = all_counts['train']
        majority = max(c.values())
        print('\nImbalance ratios (train):')
        for cls, cnt in c.items():
            print(f'  {cls}: {cnt}  (ratio vs majority: {majority/cnt:.1f}x)')


[WARN] No TrashCan annotations found — skipping class distribution plot.


## 2. TrashCan — Image Size Statistics

In [4]:
if 'train' in coco_by_split:
    coco = coco_by_split['train']
    widths  = [img['width']  for img in coco.imgs.values()]
    heights = [img['height'] for img in coco.imgs.values()]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(widths,  bins=30, color='steelblue')
    axes[0].set_title('Image Width Distribution')
    axes[1].hist(heights, bins=30, color='darkcyan')
    axes[1].set_title('Image Height Distribution')
    plt.suptitle('TrashCan — Train Image Sizes')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'trashcan_size_dist.png', dpi=150)
    plt.show()

    print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
    print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

## 3. TrashCan — Sample Images per Class

In [5]:
import random

def show_samples(coco, img_dir: Path, n_per_class: int = 3):
    cats = {c['id']: c['name'] for c in coco.loadCats(coco.getCatIds())}
    cat_ids = coco.getCatIds()
    fig, axes = plt.subplots(len(cat_ids), n_per_class,
                             figsize=(4 * n_per_class, 4 * len(cat_ids)))
    if len(cat_ids) == 1:
        axes = [axes]

    for row, cat_id in enumerate(cat_ids):
        img_ids = coco.getImgIds(catIds=[cat_id])
        sample_ids = random.sample(img_ids, min(n_per_class, len(img_ids)))
        for col, img_id in enumerate(sample_ids):
            img_info = coco.loadImgs(img_id)[0]
            img_path = img_dir / img_info['file_name']
            if not img_path.exists():
                axes[row][col].axis('off')
                continue
            img = Image.open(img_path).convert('RGB')
            axes[row][col].imshow(img)
            axes[row][col].set_title(cats[cat_id], fontsize=9)
            axes[row][col].axis('off')
    plt.suptitle('TrashCan — Sample Images per Class')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'trashcan_samples.png', dpi=100)
    plt.show()

if 'train' in coco_by_split:
    show_samples(coco_by_split['train'], TRASHCAN_ROOT / 'images' / 'train')

## 4. FLS Sonar — Class Distribution & Image Stats

In [6]:
# ── FLS Sonar — Class Distribution from PNG masks ──
if SEG_ROOT.exists():
    mask_dir = SEG_ROOT / 'Masks'
    pixel_counts = np.zeros(len(FLS_CLASSES), dtype=np.int64)
    mask_files = sorted(mask_dir.glob('*.png'))
    print(f'Found {len(mask_files)} mask files')
    for mp in mask_files:
        m = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        for cls in range(len(FLS_CLASSES)):
            pixel_counts[cls] += (m == cls).sum()

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(FLS_CLASSES, pixel_counts, color='teal')
    ax.bar_label(bars, fmt='%d', fontsize=7, rotation=90, padding=2)
    ax.set_title('FLS Sonar — Pixel Count per Class')
    ax.set_xlabel('Class')
    ax.set_ylabel('Total pixels')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('../results/eda_fls_pixel_dist.png', dpi=150)
    plt.show()

    print('\nImbalance ratios (excl. background and wall):')
    fg_counts = pixel_counts[1:11]  # classes 1-10 only
    majority = fg_counts.max()
    for i, cnt in enumerate(fg_counts, start=1):
        ratio = majority / max(cnt, 1)
        print(f'  {FLS_CLASSES[i]:20s}: {cnt:>12,}  ({ratio:.1f}x vs majority)')
else:
    print(f'[WARN] {SEG_ROOT} not found. Check data path.')


Found 1868 mask files



Imbalance ratios (excl. background and wall):
  bottle              :      691,558  (2.4x vs majority)
  can                 :      732,377  (2.3x vs majority)
  chain               :    1,341,771  (1.2x vs majority)
  drink-carton        :      251,932  (6.6x vs majority)
  hook                :      211,996  (7.9x vs majority)
  propeller           :      562,524  (3.0x vs majority)
  shampoo-bottle      :      253,656  (6.6x vs majority)
  standing-bottle     :      128,707  (13.0x vs majority)
  tire                :    1,667,687  (1.0x vs majority)
  valve               :      180,640  (9.2x vs majority)


/var/folders/0l/mrjlb_612tj_8p3bn6p6tvd00000gn/T/ipykernel_10788/499838104.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Classification datasets — image count per class ──
for name, root in [('Turntable-Cropped', TURNTABLE_ROOT), ('Watertank-Cropped', WATERTANK_ROOT)]:
    if not root.exists():
        print(f'[WARN] {root} not found')
        continue
    classes = sorted(d.name for d in root.iterdir() if d.is_dir() and not d.name.startswith('.'))
    counts = {cls: len(list((root / cls).glob('*.png'))) for cls in classes}
    total = sum(counts.values())

    fig, ax = plt.subplots(figsize=(max(6, len(classes)), 4))
    bars = ax.bar(counts.keys(), counts.values(), color='steelblue')
    ax.bar_label(bars)
    ax.set_title(f'{name} — Images per Class (total={total})')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print(f'\n{name}: {len(classes)} classes, {total} images')
    print({k: v for k, v in sorted(counts.items(), key=lambda x: -x[1])})



Turntable-Cropped: 18 classes, 4942 images
{'glass-bottle': 480, 'plastic-bottle': 440, 'glass-jar': 432, 'drink-carton': 428, 'can': 410, 'wrench': 240, 'small-tire': 234, 'plastic-propeller': 224, 'drink-sachet': 222, 'metal-bottle': 222, 'plastic-pipe': 220, 'potion-glass-bottle': 218, 'brown-glass-bottle': 210, 'large-tire': 204, 'valve': 200, 'plastic-bidon': 196, 'rotating-platform': 186, 'metal-box': 176}

Watertank-Cropped: 10 classes, 2364 images
{'bottle': 449, 'can': 367, 'drink-carton': 349, 'tire': 331, 'chain': 226, 'valve': 208, 'propeller': 137, 'hook': 133, 'shampoo-bottle': 99, 'standing-bottle': 65}


/var/folders/0l/mrjlb_612tj_8p3bn6p6tvd00000gn/T/ipykernel_10788/146760120.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. FLS Sonar — Sample Images + CLAHE Effect

In [8]:
# ── FLS Sonar — Raw vs CLAHE vs Mask samples ──
if SEG_ROOT.exists():
    img_dir  = SEG_ROOT / 'Images'
    mask_dir = SEG_ROOT / 'Masks'
    img_files = sorted(img_dir.glob('*.png'))[:4]

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cmap  = plt.colormaps['tab20'].resampled(len(FLS_CLASSES))

    fig, axes = plt.subplots(len(img_files), 3, figsize=(12, 4 * len(img_files)))
    for row, img_path in enumerate(img_files):
        raw  = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        enhanced = clahe.apply(raw)
        mask_path = mask_dir / img_path.name
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else None

        axes[row, 0].imshow(raw, cmap='gray')
        axes[row, 0].set_title('Raw sonar')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(enhanced, cmap='gray')
        axes[row, 1].set_title('After CLAHE')
        axes[row, 1].axis('off')

        if mask is not None:
            axes[row, 2].imshow(mask, cmap=cmap, vmin=0, vmax=len(FLS_CLASSES))
            axes[row, 2].set_title('Pixel mask')
        else:
            axes[row, 2].set_title('Mask not found')
        axes[row, 2].axis('off')

    plt.suptitle('FLS Sonar — Raw vs CLAHE vs Mask')
    plt.tight_layout()
    plt.savefig('../results/eda_fls_clahe.png', dpi=100)
    plt.show()
else:
    print(f'[WARN] {SEG_ROOT} not found')


/var/folders/0l/mrjlb_612tj_8p3bn6p6tvd00000gn/T/ipykernel_10788/3065460615.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Summary & Preprocessing Decisions

| Dataset | Key Finding | Action |
|---|---|---|
| TrashCan | `trash` class dominates | Weighted sampler + focal loss |
| TrashCan | Variable image sizes | Resize to 640×640 with padding |
| FLS | Low-contrast speckle images | CLAHE + Gaussian blur σ=1 |
| FLS | Long-tail class distribution | Weighted sampler + Dice loss |